
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# 메달리온 아키텍처를 사용하여 데이터 변환
![](https://velog.velcdn.com/images/newnew_daddy/post/c04c1f10-3d64-4373-a9da-8edfd929dc40/image.png)

## 환경 준비 (Databricks Free Edition)

- **카탈로그** `db_catalog`, **스키마** `db_schema` 를 사용합니다.
- 스키마 `db_schema` 아래 **`myfiles`** 볼륨과 **employees.csv** (및 브론즈 실습용 **employees2.csv**)를 준비합니다. employees2는 이전 노트북(수집)에서 생성하거나 동일 형식으로 업로드합니다.

다음 셀에서 `USE CATALOG` / `USE SCHEMA` 를 실행합니다.

In [0]:
%sql
USE CATALOG db_catalog;
USE SCHEMA db_schema;

## A. 환경 구성 및 탐색

1. 기본 카탈로그를 **db_catalog**로 설정하고 스키마를 특정 스키마로 설정합니다. 그런 다음 현재 카탈로그 및 스키마를 봅니다.

In [0]:
%sql
USE CATALOG db_catalog;
USE SCHEMA db_schema;

SELECT current_catalog(), current_schema()

2. 스키마의 **myfiles** 볼륨에서 사용 가능한 파일을 봅니다. 볼륨에 두 개의 CSV 파일, **employees.csv** 및 **employees2.csv**가 포함되어 있는지 확인합니다.

In [0]:
spark.sql(f"LIST '/Volumes/db_catalog/db_schema/myfiles/' ").display()

## B. 메달리온 아키텍처의 간단한 예

**목적:** 일정에 따라 실행되도록 예약할 수 있는 작업을 만듭니다. 파이프라인은 다음을 수행합니다.

1. **myfiles** 볼륨에서 모든 CSV 파일을 수집하고 브론즈 테이블을 만듭니다.
2. 새 열을 추가하여 브론즈 테이블을 준비하고 실버 테이블을 만듭니다.
3. 소비자를 위한 골드 집계 테이블을 만듭니다.

### BRONZE
**목표:** **myfiles** 볼륨의 모든 CSV 파일을 사용하여 테이블을 만듭니다.

1. 셀을 실행하여 다음 단계를 수행합니다.

    - 문은 이미 존재하는 경우 테이블을 `DROP TABLE IF EXISTS`**current_employees_bronze** 삭제합니다(데모 목적으로).

    - `CREATE TABLE IF NOT EXISTS` 문은 Delta 테이블 **current_employees_bronze** 이 아직 존재하지 않는 경우 테이블을 만들고 해당 열을 정의합니다.

    - `COPY INTO` 문:
        - 스키마의 **myfiles** 볼륨에서 테이블로 **current_employees_bronze** 모든 CSV 파일을 로드합니다.
        - 첫 번째 행을 헤더로 사용하고 CSV 파일에서 스키마를 유추합니다.

    - 최종 `SELECT` 쿼리는 **current_employees_bronze** 테이블의 모든 행을 표시합니다.


결과를 보고 테이블에 **6개의 행**과 **4개의 열**이 포함되어 있는지 확인합니다.


In [0]:
%sql
-- 데모 목적으로 존재하는 경우 테이블을 삭제합니다.
DROP TABLE IF EXISTS current_employees_bronze;

-- 빈 테이블 및 열 만들기
CREATE TABLE IF NOT EXISTS current_employees_bronze (
  ID INT,
  FirstName STRING,
  Country STRING,
  Role STRING
  );

In [0]:
## 브론즈 원시 수집 테이블을 만들고 행에 대한 CSV 파일 이름을 포함합니다.
spark.sql(f'''
  COPY INTO current_employees_bronze
  FROM '/Volumes/db_catalog/db_schema/myfiles/'
  FILEFORMAT = CSV
  FORMAT_OPTIONS (
    'header' = 'true', 
    'inferSchema' = 'true'
)
''').display()

In [0]:
%sql
SELECT * 
FROM current_employees_bronze;

### SILVER
**목적:** 브론즈 테이블을 변환하고 실버 테이블을 만듭니다.

1. **current_employees_silver** 테이블에서 **current_employees_bronze**라는 이름의 테이블을 만듭니다. 

    테이블은 다음을 수행합니다.
    - **ID**, **FirstName**, **Country** 열을 선택합니다.
    - **Role** 열을 대문자로 변환합니다.
    - 두 개의 새 열을 추가합니다. **CurrentTimeStamp** 및 **CurrentDate**.

In [0]:
%sql
-- 데이터를 최종 실버 테이블에 병합하는 데 사용할 임시 뷰 만들기
CREATE OR REPLACE TABLE current_employees_silver AS 
SELECT 
  ID,
  FirstName,
  Country,
  upper(Role) as Role,                 -- 역할 열 대문자로 변경
  current_timestamp() as CurrentTimeStamp,    -- 현재 날짜/시간 가져오기
  date(CurrentTimeStamp) as CurrentDate       -- 날짜 확인하기
FROM current_employees_bronze;

2. 테이블을 **current_employees_silver** 봅니다. 테이블에 6개의 행과 6개의 열이 포함되어 있는지 확인합니다.

In [0]:
%sql
SELECT * 
FROM current_employees_silver;

### GOLD
**목표:** 실버 테이블을 집계하여 최종 골드 테이블을 만듭니다.

1. **temp_view_total_roles**라는 이름의 임시 뷰를 만들고 역할별로 총 직원 수를 집계합니다. 그런 다음 보기 결과를 표시합니다.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW temp_view_total_roles AS 
SELECT
  Role, 
  count(*) as TotalEmployees
FROM current_employees_silver
GROUP BY Role;


SELECT *
FROM temp_view_total_roles;

2. **total_roles_gold**라는 이름의 최종 골드 테이블을 지정된 열로 만듭니다.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS total_roles_gold (
  Role STRING,
  TotalEmployees INT
);

3. 집계된 임시 뷰 **temp_view_total_roles**의 모든 행을 **total_roles_gold** 테이블에 삽입하여 테이블의 기존 데이터를 덮어씁니다. 이 작업은 테이블의 데이터를 덮어쓰지만 기존 스키마와 테이블 정의 및 속성은 유지합니다.

    다음을 확인합니다.
    - **num_affected_rows** 는 *4*입니다.
    - **num_inserted_rows** 는 *4*입니다.

In [0]:
%sql
INSERT OVERWRITE TABLE total_roles_gold
SELECT * 
FROM temp_view_total_roles;

4. **total_roles_gold** 테이블을 쿼리하여 역할별 총 직원 수를 봅니다.

In [0]:
%sql
SELECT *
FROM total_roles_gold;

5. **total_roles_gold** 테이블의 기록을 봅니다.

In [0]:
%sql
DESCRIBE HISTORY total_roles_gold;

## C. 데이터 거버넌스 및 보안
**목표:** 테이블의 **total_roles_gold** 리니지를 보고 해당 권한을 설정하는 방법을 알아봅니다.

### 1. 다음을 완료하여 **카탈로그 탐색기**에서 스키마를 엽니다.

   - a. 왼쪽 탐색 모음에서 카탈로그 아이콘을 선택합니다. 

   - b. 검색 창(**db_catalog**)에 모듈의 카탈로그 이름을 입력합니다.

   - c. 새로 고침 아이콘을 선택하여 **db_catalog** 카탈로그를 새로 고칩니다.

   - d. **db_catalog** 카탈로그를 확장합니다. 카탈로그 내에 다양한 스키마(데이터베이스)가 표시되어야 합니다.

   - e. 스키마를 찾아 선택합니다. 첫 번째 셀의 설정 노트에서 스키마를 찾을 수 있습니다. 

   -  f. 스키마 오른쪽에 있는 옵션 아이콘을 클릭하고 **카탈로그 탐색기에서 열기를** 선택합니다.

   - g. 데모에서 생성한 네 개의 테이블 (**current_employees_bronze**, **current_employees_silver**, **total_roles_gold**, **wine_quality_table**)이 해당 스키마의 카탈로그 탐색기에 표시됨을 확인하십시오.

   - h. **카탈로그 탐색기**에서 **total_roles_gold** 테이블을 선택합니다.

**카탈로그 탐색기** 탭을 열어 둡니다.

### 2. 카탈로그 탐색기에서 **total_roles_gold** 테이블의 권한, 이력, 리니지 및 인사이트를 보려면 다음을 완료하십시오. 

   a. **권한.** 

  - 권한 **탭을** 선택합니다. 그러면 테이블에 대한 모든 권한이 표시됩니다. 현재 테이블에는 권한이 설정되어 있지 않습니다.

  - **부여**를 선택합니다. 이를 통해 여러 보안 주체를 추가하고 권한을 할당할 수 있습니다. 사용자는 테이블의 카탈로그 및 스키마에 대한 액세스 권한이 있어야 합니다.

  - **취소**를 선택합니다. 

   b. **히스토리**

  - **히스토리** 탭을 선택합니다. 그러면 테이블의 이력이 표시됩니다. **total_roles_gold** 테이블은 현재 두 가지 버전이 있습니다.

   c. **리니지**

  - **리니지** 탭을 선택합니다. 그러면 테이블의 리니지가 표시됩니다. 테이블이 **current_employees_silver** 표시되는지 확인합니다.

  - **리니지 그래프 보기 버튼**을 선택합니다. 이렇게 하면 테이블의 리니지가 시각적으로 표시됩니다. 아이콘을 선택하여 추가 정보를 볼 수 있습니다.

  - 리니지 그래프를 닫습니다.

   d. **인사이트**

  - **인사이트** 탭을 선택합니다. **카탈로그 탐색기**의 인사이트 탭 을 사용하여 Unity Catalog에 등록된 테이블의 가장 빈번한 최근 쿼리 및 사용자를 볼 수 있습니다. 인사이트 탭은 지난 30일 동안의 빈번한 쿼리 및 사용자 액세스에 대해 보고합니다.

   e. **카탈로그 탐색기** 브라우저 탭을 닫습니다.

##D. 정리
1. 뷰 및 테이블을 삭제합니다.

In [0]:
%sql
-- 테이블을 삭제합니다.
DROP TABLE IF EXISTS current_employees_bronze;
DROP TABLE IF EXISTS current_employees_silver;
DROP TABLE IF EXISTS total_roles_gold;

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>